# BioGPT Foundation LLM
Fine-tuned on `ChatDoctor_HealthCareMagic_112k` using LoRA.

Preprocessing pipeline was created by Aaron.

In [ ]:
!pip install torch transformers accelerate peft datasets trl bitsandbytes sacremoses
!pip install --upgrade "torchao>=0.16.0"
!pip install contractions textblob spacy
!python -m textblob.download_corpora lite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import re
import contractions
from textblob import TextBlob
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import load_dataset, DatasetDict

assert torch.cuda.is_available(), "CUDA not available. Please enable GPU."
print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: Tesla T4


## Preprocessing Pipeline

- Clean punctuation and regex noise while preserving clinical abbreviations and decimals.
- Expand contractions and protect medical terms during typo correction.
- Normalize text with ClinicalBERT, then run the full pipeline in order.

In [ ]:
def clean_punctuation(text):
    # Clean punctuation while preserving clinical abbreviations and decimals.

    if not isinstance(text, str):
        return ""

    text = text.lower()

    # Preserve clinical abbreviations and decimal numbers.
    text = re.sub(r'i\.e\.', '_LOCK_IE_', text)
    text = re.sub(r'\bi\.e\b', '_LOCK_IE_', text)
    text = re.sub(r'e\.g\.', '_LOCK_EG_', text)
    text = re.sub(r'\be\.g\b', '_LOCK_EG_', text)
    text = re.sub(r'dr\.', '_LOCK_DR_', text)

    text = re.sub(r'[\(\)]', ' ', text)
    text = re.sub(r'([a-z])(\d)', r'\1 \2', text)
    text = re.sub(r'(\d)([a-z])', r'\1 \2', text)
    text = text.replace('?', ' ')
    text = re.sub(r'(\d+)\.(\d+)', r'\1_SAFE_DEC_\2', text)
    text = re.sub(r'\.{2,}', ' ', text)
    text = re.sub(r'([a-z0-9])\.([a-z])', r'\1 \2', text)
    text = re.sub(r'[.,;!]+(?=\s|$)', '', text)
    text = re.sub(r'\s\.\s', ' ', text)

    text = text.replace('_LOCK_IE_', 'i.e')
    text = text.replace('_LOCK_EG_', 'e.g')
    text = text.replace('_LOCK_DR_', 'dr.')
    text = text.replace('_SAFE_DEC_', '.')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def expand_contractions(text):
    # Expand English contractions with the contractions library.
    if not isinstance(text, str):
        return ""
    text = contractions.fix(text)
    return re.sub(r'\s+', ' ', text).strip()


print("Loading ClinicalBERT tokenizer for normalization...")
clinical_tokenizer = AutoTokenizer.from_pretrained("medicalai/ClinicalBERT")

# Preserve medical terms that should not be autocorrected.
MEDICAL_SAFETY_SHIELD = {
    'panadol', 'nurofen', 'pooing', 'weezy', 'croupy', 'raspy',
    'rattly', 'uti', 'ild', 'hpylori', 'gyno', 'cause'
}

def autocorrect_typos(text):
    
    # Fix known typos while skipping medical safety terms.
    words = text.split()
    corrected = []
    for word in words:
        clean_word = re.sub(r'[^a-z0-9]', '', word.lower())
        if clean_word in MEDICAL_SAFETY_SHIELD:
            corrected.append(word)
        elif clean_word in ("immediatly", "afect", "experiencing"):
            corrected.append(str(TextBlob(word).correct().lower()))
        else:
            corrected.append(word)
    return " ".join(corrected)

def normalize_with_clinicalbert(text):

    # Normalize medical text with ClinicalBERT tokenization.
    if not isinstance(text, str):
        return ""
    text = expand_contractions(text)
    text = autocorrect_typos(text)
    tokens = clinical_tokenizer.tokenize(text)
    clean_text = clinical_tokenizer.convert_tokens_to_string(tokens)
    clean_text = clean_text.replace(" .", ".").replace(" ,", ",").strip()
    return clean_text


# Run cleaning, contraction expansion, and tokenizer normalization in sequence.
def full_preprocess(text):

    text = clean_punctuation(text)
    text = normalize_with_clinicalbert(text)
    return text

print("Preprocessing pipeline ready.")

sample = "I've been having chest pain...it's really bad!!! Dr. Smith said my temp is 98.6"
print(f"\nOriginal : {sample}")
print(f"Processed: {full_preprocess(sample)}")

Loading ClinicalBERT tokenizer for normalization...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Preprocessing pipeline ready.

Original : I've been having chest pain...it's really bad!!! Dr. Smith said my temp is 98.6
Processed: i have been having chest pain it is really bad dr. smith said my temp is 98. 6


## Model Loading & LoRA Configuration

In [ ]:
model_name = "microsoft/biogpt"

# biogpt_tokenizer for training
biogpt_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if biogpt_tokenizer.pad_token is None:
    biogpt_tokenizer.pad_token = biogpt_tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map={"":0},
    trust_remote_code=True
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/595 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/927k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/696k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie biogpt.embed_tokens.weight to output_projection.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 3,145,728 || all params: 393,310,208 || trainable%: 0.7998


In [ ]:
# Load Dataset & Apply Preprocessing
raw_dataset = load_dataset("xDAN-datasets/ChatDoctor_HealthCareMagic_112k", split="train")
raw_dataset = raw_dataset.select(range(5000))
print(f"Total samples: {len(raw_dataset)}")

system_prompt = (
    "You are an experienced family doctor giving clear, direct medical advice. "
    "Answer in second person. Do not roleplay as a patient."
)

# Patterns compiled once outside the function for efficiency
OPEN_PAT = re.compile(
    r'^(I have gone through[^.]*\.|Thanks for your[^.]*\.|I am Chat[^.]*\.|'
    r'I gone through[^.]*\.|Welcome to[^.]*\.|Hello[^.]*\.|Hi[^.]*\.)\s*',
    flags=re.IGNORECASE
)
TRAIL_PAT = re.compile(
    r'\s*(Hope\b.*|Wish\b.*|Take care\b.*|Kindly\b.*|Please (?:consult|advise|help)\b.*|'
    r'Let me know\b.*|Thanks for using\b.*|Chat\s?[Dd]octor\b.*)',
    flags=re.IGNORECASE | re.DOTALL
)

def format_chat(example):
    convs = example["conversations"]
    if len(convs) != 2 or convs[0]["from"] != "human" or convs[1]["from"] != "gpt":
        return {"text": ""}

    user_msg      = convs[0]["value"].strip()
    assistant_msg = convs[1]["value"].strip()

    # Apply teammate preprocessing pipeline to user message
    user_msg = full_preprocess(user_msg)

    # Clean assistant response (greeting + trailing noise)
    assistant_clean = OPEN_PAT.sub("", assistant_msg).strip()
    assistant_clean = TRAIL_PAT.sub("", assistant_clean).strip()
    
    # Replace collegial "we" with patient-directed "you"
    assistant_clean = re.sub(r'\bwe\b', 'you', assistant_clean, flags=re.IGNORECASE)
    assistant_clean = re.sub(r'\bour\b', 'your', assistant_clean, flags=re.IGNORECASE)
    assistant_clean = assistant_clean[:400]

    # Drop samples where cleaning removed all content
    if len(assistant_clean) < 30:
        return {"text": ""}

    text = f"{system_prompt}\nUser: {user_msg}\nAssistant: {assistant_clean}\n"
    return {"text": text}

print("Applying preprocessing pipeline to dataset (this may take a few minutes)...")
dataset = raw_dataset.map(format_chat, remove_columns=raw_dataset.column_names)
dataset = dataset.filter(lambda x: x["text"] != "")
print(f"After preprocessing & cleaning: {len(dataset)} samples")

# 90/10 train/eval split
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

README.md:   0%|          | 0.00/612 [00:00<?, ?B/s]

data/train-00000-of-00001-01a45afed1b713(…):   0%|          | 0.00/141M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

Total samples: 5000
Applying preprocessing pipeline to dataset (this may take a few minutes)...


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

After preprocessing & cleaning: 4600 samples
Train: 4140 | Eval: 460


In [ ]:
# Show 3 samples to verify preprocessing is working correctly
print("Sample preprocessed training examples:\n")
for i in range(3):
    print(f"--- Sample {i+1} ---")
    print(train_dataset[i]["text"][:400])
    print()

Sample preprocessed training examples:

--- Sample 1 ---
You are an experienced family doctor giving clear, direct medical advice. Answer in second person. Do not roleplay as a patient.
User: my dad is 74 his kidneys are functioning at 6 % he has enlarged prostrate glands but the doctors are not concerned of this at the moment he is not on dialysis and is doing quite well for a person who is kidney function is at 6 % can prostrate glands affect the kidn

--- Sample 2 ---
You are an experienced family doctor giving clear, direct medical advice. Answer in second person. Do not roleplay as a patient.
User: i am 40 yrs old female wt = 65 kg ht = 5 ft 2 in i have two daughters my husband is also a teacher i teach in high school my problem is knee pain and some times pain in middle fingers quite often and pain in my leg below the knees i feel that i am lossing my memory

--- Sample 3 ---
You are an experienced family doctor giving clear, direct medical advice. Answer in second person. Do not

## Tokenization for training

In [ ]:
def tokenize_fn(examples):
    tokenized = biogpt_tokenizer(
        examples["text"],
        truncation=True,
        padding=True,
        max_length=512,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_eval  = eval_dataset.map(tokenize_fn,  batched=True, remove_columns=["text"])

tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_eval.set_format(type="torch",  columns=["input_ids", "attention_mask", "labels"])

print("Tokenization done. Example shape:", tokenized_train[0]["input_ids"].shape)

Map:   0%|          | 0/4140 [00:00<?, ? examples/s]

Map:   0%|          | 0/460 [00:00<?, ? examples/s]

Tokenization done. Example shape: torch.Size([512])


## Training

In [ ]:
training_args = TrainingArguments(
    output_dir="./biogpt-familydoctor",  
    per_device_train_batch_size=1,  # Keep GPU memory usage low
    gradient_accumulation_steps=8,  
    num_train_epochs=3,
    learning_rate=2e-4,  
    fp16=True,  
    logging_steps=50,
    save_steps=400,  # Periodic checkpoint saving
    eval_strategy="steps",
    eval_steps=200,
    load_best_model_at_end=True,  # Restore best checkpoint after training
    metric_for_best_model="eval_loss",
    warmup_steps=100,
    report_to="none",
    gradient_checkpointing=True,  # Reduce memory during backprop
    optim="adafactor",
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=biogpt_tokenizer,
    mlm=False,  # Causal LM training
    pad_to_multiple_of=8
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]  # Stop if eval loss stops improving
)

trainer.train()

model.save_pretrained("./biogpt-familydoctor-final")
biogpt_tokenizer.save_pretrained("./biogpt-familydoctor-final")
print("Training complete and model saved.")

Step,Training Loss,Validation Loss
200,3.419869,3.326625
400,3.312659,3.211501
600,3.257446,3.165711
800,3.223933,3.135269
1000,3.175005,3.115948
1200,3.159872,3.101526
1400,3.127538,3.093187


Training complete and model saved.


## Inference/Testing

In [ ]:
import gc
del model, trainer
gc.collect()
torch.cuda.empty_cache()

base_model_name = "microsoft/biogpt"
lora_path = "./biogpt-familydoctor-final"

biogpt_tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
if biogpt_tokenizer.pad_token is None:
    biogpt_tokenizer.pad_token = biogpt_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map={"":0},
    trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, lora_path)
model.eval()
print("Model loaded successfully.")

def generate_answer(question, max_new_tokens=150):
   
    # Generate a concise answer using greedy decoding. Input question is preprocessed using the same pipeline as training data.
    system_prompt = (
        "You are an experienced family doctor giving clear, direct medical advice. "
        "Answer in second person directly to the patient. "
        "Do not start with greetings. Do not roleplay as a patient."
    )

    # Apply same preprocessing to question at inference time
    clean_question = full_preprocess(question)
    prompt = f"{system_prompt}\nUser: {clean_question}\nAssistant:"

    inputs = biogpt_tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.3,
            early_stopping=True,
            pad_token_id=biogpt_tokenizer.pad_token_id,
            eos_token_id=biogpt_tokenizer.eos_token_id,
        )

    full_output = biogpt_tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Assistant:" in full_output:
        assistant_part = full_output.split("Assistant:")[-1]
    else:
        assistant_part = full_output

    # Post-process: strip any remaining greeting patterns
    assistant_part = OPEN_PAT.sub("", assistant_part.strip())
    assistant_part = TRAIL_PAT.sub("", assistant_part).strip()

    return assistant_part.strip()

# Test
test_questions = [
    "What are the early symptoms of a heart attack?",
    "How can I treat a minor burn at home?",
    "What is the difference between Type 1 and Type 2 diabetes?",
    "Is it safe to take ibuprofen during pregnancy?",
    "How often should adults get a flu shot?"
]

print("\n" + "="*80)
print("TESTING FINETUNED BIOGPT HEALTHCARE ASSISTANT")
print("="*80)

for i, q in enumerate(test_questions, 1):
    print(f"\n[{i}] Question: {q}")
    answer = generate_answer(q, max_new_tokens=120)
    print(f"Answer: {answer}")
    print("-"*80)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie biogpt.embed_tokens.weight to output_projection.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model loaded successfully.

TESTING FINETUNED BIOGPT HEALTHCARE ASSISTANT

[1] Question: What are the early symptoms of a heart attack?
Answer: I can understand your concern and concerns went through your details. It is very important for you to know about this disease so that it may be diagnosed earlier by doctors or other health care professionals. In my opinion, there should be no need to worry much about cardiac diseases like coronary artery disease (CAD) because they do have some risk factors such as hypertension, diabetes mellitus etc.. This will help us diagnose these patients sooner. So if you had any history of angina pectoris then go on to get done ECG examination which would reveal whether the condition was due to CAD or another cause. If the diagnosis is correct
--------------------------------------------------------------------------------

[2] Question: How can I treat a minor burn at home?
Answer: I understand your concern and concerns went through your details. It is v

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# import os
# import shutil

# drive_path = '/content/drive/MyDrive/biogpt_familydoctor_final'
# os.makedirs(drive_path, exist_ok=True)

# source_path = './biogpt-familydoctor-final'

# # Copy all files from the source directory to the drive directory
# for item_name in os.listdir(source_path):
#     s = os.path.join(source_path, item_name)
#     d = os.path.join(drive_path, item_name)
#     if os.path.isdir(s):
#         shutil.copytree(s, d, dirs_exist_ok=True)
#     else:
#         shutil.copy2(s, d)

# print(f"Model saved to {drive_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model saved to /content/drive/MyDrive/biogpt_familydoctor_final
